# 02 — VP-SGLD 하이퍼파라미터 Sweep

nb 01 이 만든 ensemble (`ENSEMBLE_ROOT`) 위에서 VP-SGLD sampler 내부 파라미터만 변경해가며
샘플 품질 비교. OAT (one-at-a-time) — 한 번에 한 축만 sweep, 나머지는 `FIXED`.

### sweep 가능한 축 (전부 VP-SGLD 내부, ensemble 은 건드리지 않음)

| axis | 역할 | paper 연결 |
|---|---|---|
| `beta` | preconditioner 강도 `M = (I + β Σ)^{-1}` | **§6 Ablation 1** (M=I 제거) 필수 |
| `eta` | step size `η_t` | MCMC 튜닝 |
| `n_steps` | SGLD chain 길이 | convergence |
| `tau` | noise 온도 `τ_0` | temperature |
| `sigma_start` | init 섭동 std | init 위치 영향 |
| `auto_beta` | β 자동 스케일 (median Σ 기준) | dimensionless β |
| `restart` | §3.3 restart rule on/off | optional |

ensemble 속성 (K, methods, ...) 은 nb 01 에서 정함. 비교하려면 nb 01 을 여러 번 돌려 여러 ROOT 생성.

## 0. 설정 — `ARGS_STR` 하나로 모든 sweep/fixed/실행환경 제어

셀 2 의 `ARGS_STR` 문자열이 **CLI 처럼** 파싱됨. 편집해서:
- `--beta 100 --eta 0.03` → FIXED 값 덮어씀
- `--sweep-beta 0.0 10.0 100.0` → beta axis sweep 활성화
- `--single-gpu 0` → GPU 한 개로 순차 실행 (병렬 off)
- `--classes 0 1`, `--top-n 5`, `--rank-metric sample_quality` 등

주석 (`#` 뒤) 과 공백은 자동 제거. SWEEPS 는 원하는 축의 `--sweep-*` 주석 풀면 활성화.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
os.chdir('/home/work/JooKyung/TabEBM')

import sys, json, warnings, time, itertools, argparse, shlex, datetime as _dt
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings('ignore')
sys.path.insert(0, 'experiments'); sys.path.insert(0, 'src')

import numpy as np, pandas as pd, torch
from pathlib import Path
import matplotlib.pyplot as plt

from tabebm.vp_sgld import vp_sgld_from_ensemble

pd.set_option('display.width', 220); pd.set_option('display.precision', 4)

# ======================================================================
#  CLI-style 설정 — 이 문자열만 수정하면 모든 FIXED/SWEEPS/실행환경 제어.
#  한 줄당 하나의 flag. '#' 이후는 주석. 주석 해제/추가로 axis 활성화.
# ======================================================================
ARGS_STR = '''
    # --- 대상 ensemble ---
    # --ensemble-root experiments/ebms/20260415_210502_Distance_EBM
    # (생략 시 experiments/ebms/ 에서 가장 최근 *_EBM auto-detect)

    # --- FIXED (한 run 의 default) ---
    --beta 1.0
    --eta 0.05
    --tau 1.0
    --sigma-start 0.1
    --n-steps 50
    --n-samples 500
    --seed 0
    --auto-beta
    # --restart
    # --kappa-sigma 0.5
    # --kappa-mu 0.2
    --ignore-variance            # M=I 로 고정 (variance preconditioner off → plain SGLD-like)

    # --- SWEEPS (주석 풀면 해당 axis 활성화) ---
    # --sweep-beta 0.0 0.1 1.0 10.0 100.0
    # --sweep-eta 0.01 0.03 0.05 0.1
    # --sweep-tau 0.0 0.5 1.0 2.0
    # --sweep-sigma-start 0.05 0.1 0.3
    # --sweep-n-steps 20 50 100 200
    # --sweep-auto-beta
    # --sweep-restart
    # --sweep-ignore-variance       # [False, True] 비교

    # --- 실행환경 ---
    --classes 0 1
    --gpus 0 1 2 3
    # --single-gpu 0      # 설정 시 이 GPU 하나에서만 **순차** 실행 (gpus 덮어씀)

    # --- wandb (전부 로컬 PNG + wandb Image 로 중복 업로드) ---
    # --wandb-project tabebm-sweep
    # --wandb-run-name my_run       # 생략 시 timestamp 기반
    # --wandb-mode offline          # online / offline / disabled

    # --- Top-N 선정 ---
    --top-n 5
    --rank-metric sample_quality
    # --rank-descending     # 기본은 오름차순 (낮을수록 좋음)
'''

# ======================================================================
#  parser 정의
# ======================================================================
parser = argparse.ArgumentParser(add_help=False)
# ensemble
parser.add_argument('--ensemble-root', type=Path, default=None)
# fixed
parser.add_argument('--beta', type=float, default=1.0)
parser.add_argument('--eta', type=float, default=0.05)
parser.add_argument('--tau', type=float, default=1.0)
parser.add_argument('--sigma-start', type=float, default=0.1)
parser.add_argument('--n-steps', type=int, default=50)
parser.add_argument('--n-samples', type=int, default=500)
parser.add_argument('--seed', type=int, default=0)
parser.add_argument('--auto-beta', dest='auto_beta', action='store_true', default=False)
parser.add_argument('--no-auto-beta', dest='auto_beta', action='store_false')
parser.add_argument('--restart', dest='restart', action='store_true', default=False)
parser.add_argument('--kappa-sigma', type=float, default=None)
parser.add_argument('--kappa-mu', type=float, default=None)
parser.add_argument('--ignore-variance', dest='ignore_variance',
                     action='store_true', default=False)
# sweeps
parser.add_argument('--sweep-beta', type=float, nargs='+', default=None)
parser.add_argument('--sweep-eta', type=float, nargs='+', default=None)
parser.add_argument('--sweep-tau', type=float, nargs='+', default=None)
parser.add_argument('--sweep-sigma-start', type=float, nargs='+', default=None)
parser.add_argument('--sweep-n-steps', type=int, nargs='+', default=None)
parser.add_argument('--sweep-n-samples', type=int, nargs='+', default=None)
parser.add_argument('--sweep-auto-beta', action='store_true', default=False,
                     help='auto_beta 를 [False, True] 로 sweep')
parser.add_argument('--sweep-restart', action='store_true', default=False,
                     help='restart 를 [False, True] 로 sweep')
parser.add_argument('--sweep-ignore-variance', action='store_true', default=False,
                     help='ignore_variance 를 [False, True] 로 sweep')
# execution
parser.add_argument('--classes', type=int, nargs='+', default=[0, 1])
parser.add_argument('--gpus', type=int, nargs='+', default=[0, 1, 2, 3])
parser.add_argument('--single-gpu', type=int, default=None)
# top-N
parser.add_argument('--top-n', type=int, default=5)
parser.add_argument('--rank-metric', default='sample_quality')
parser.add_argument('--rank-descending', action='store_true', default=False)
# wandb
parser.add_argument('--wandb-project', type=str, default=None,
                     help='설정하지 않으면 wandb 업로드 비활성')
parser.add_argument('--wandb-run-name', type=str, default=None)
parser.add_argument('--wandb-mode', type=str, default='online',
                     choices=['online', 'offline', 'disabled'])

# --- ARGS_STR 를 argv 로 변환 (주석 제거 + shlex) ---
argv = []
for line in ARGS_STR.splitlines():
    line = line.split('#', 1)[0].strip()
    if line:
        argv.extend(shlex.split(line))
args = parser.parse_args(argv)

# ======================================================================
#  ENSEMBLE_ROOT 결정
# ======================================================================
if args.ensemble_root is None:
    cand = sorted(p for p in Path('experiments/ebms').iterdir()
                   if p.is_dir() and p.name.endswith('_EBM'))
    assert cand, 'No *_EBM/ run found — run nb 01 first.'
    ENSEMBLE_ROOT = cand[-1]
    print(f'[auto] ENSEMBLE_ROOT = {ENSEMBLE_ROOT}')
else:
    ENSEMBLE_ROOT = args.ensemble_root
assert (ENSEMBLE_ROOT / 'c0' / 'meta.json').exists(), \
    f'no c0/meta.json under {ENSEMBLE_ROOT}'

# ======================================================================
#  SWEEPS / FIXED 재구성 (downstream 셀은 그대로 이 이름 사용)
# ======================================================================
SWEEPS = {}
if args.sweep_beta:         SWEEPS['beta']        = args.sweep_beta
if args.sweep_eta:          SWEEPS['eta']         = args.sweep_eta
if args.sweep_tau:          SWEEPS['tau']         = args.sweep_tau
if args.sweep_sigma_start:  SWEEPS['sigma_start'] = args.sweep_sigma_start
if args.sweep_n_steps:      SWEEPS['n_steps']     = args.sweep_n_steps
if args.sweep_n_samples:    SWEEPS['n_samples']   = args.sweep_n_samples
if args.sweep_auto_beta:    SWEEPS['auto_beta']   = [False, True]
if args.sweep_restart:      SWEEPS['restart']     = [False, True]
if args.sweep_ignore_variance: SWEEPS['ignore_variance'] = [False, True]

FIXED = {
    'beta':        args.beta,
    'eta':         args.eta,
    'n_steps':     args.n_steps,
    'tau':         args.tau,
    'sigma_start': args.sigma_start,
    'auto_beta':   args.auto_beta,
    'restart':     args.restart,
    'kappa_sigma': args.kappa_sigma,
    'kappa_mu':    args.kappa_mu,
    'ignore_variance': args.ignore_variance,
    'n_samples':   args.n_samples,
    'seed':        args.seed,
}

CLASSES = args.classes
# --single-gpu 가 주어지면 GPUS = [그 GPU 하나] 로 강제 → 이후 셀 7 이 자동 sequential
if args.single_gpu is not None:
    GPUS = [args.single_gpu]
    PARALLEL = False
    print(f'[single-gpu] GPU {args.single_gpu} 만 사용, sweep 순차 실행')
else:
    GPUS = args.gpus
    PARALLEL = len(GPUS) > 1

METRICS        = ['sample_quality', 'diagnostics']
TOP_N          = args.top_n
RANK_METRIC    = args.rank_metric
RANK_ASCENDING = not args.rank_descending

# ======================================================================
#  SWEEP_OUT_DIR
# ======================================================================
_sweep_tag = '-'.join(sorted(SWEEPS)) or 'baseline_only'
SWEEP_OUT_DIR = ENSEMBLE_ROOT / 'sweeps' / f'{_dt.datetime.now().strftime("%Y%m%d_%H%M%S")}_{_sweep_tag}'
SWEEP_OUT_DIR.mkdir(parents=True, exist_ok=True)

(SWEEP_OUT_DIR / 'config.json').write_text(json.dumps({
    'ensemble_root': str(ENSEMBLE_ROOT),
    'ARGS_parsed': vars(args),
    'SWEEPS': SWEEPS, 'FIXED': FIXED, 'METRICS': METRICS,
    'TOP_N': TOP_N, 'RANK_METRIC': RANK_METRIC, 'RANK_ASCENDING': RANK_ASCENDING,
    'CLASSES': CLASSES, 'GPUS': GPUS, 'PARALLEL': PARALLEL,
}, indent=2, default=str))
print(f'Sweep outputs → {SWEEP_OUT_DIR}')

# ======================================================================
#  요약
# ======================================================================
run_meta = json.loads((ENSEMBLE_ROOT / 'meta.json').read_text())
print(f'\nRun methods: {run_meta["methods"]}  K={run_meta["n_ebms"]}')
print(f'FIXED  : {FIXED}')
print(f'SWEEPS : {SWEEPS if SWEEPS else "(none — baseline only)"}')
print(f'CLASSES={CLASSES}  GPUS={GPUS}  PARALLEL={PARALLEL}')
print(f'total runs: {1 + sum(len(v) for v in SWEEPS.values())}  (1 baseline + sweep values)')

# ======================================================================
#  SWEEPS_EXTRA — dict 스타일로 직접 sweep 축 추가/덮어쓰기
#  ARGS_STR 의 '--sweep-*' 가 안 보이고 답답할 때 여기 한 줄 주석 풀면 끝.
#  ARGS_STR 보다 우선 적용 (같은 axis 있으면 여기 값이 이김).
# ======================================================================
SWEEPS_EXTRA = {
    # 'beta':        [0.0, 0.1, 1.0, 10.0, 100.0],
    # 'eta':         [0.01, 0.03, 0.05, 0.1],
    # 'tau':         [0.0, 0.5, 1.0, 2.0],
    # 'sigma_start': [0.05, 0.1, 0.3],
    # 'n_steps':     [20, 50, 100, 200],
    # 'n_samples':   [100, 500, 1000],
    # 'auto_beta':   [False, True],
    # 'restart':     [False, True],
    # 'ignore_variance': [False, True],
}
SWEEPS.update(SWEEPS_EXTRA)

# SWEEPS_EXTRA 적용된 최종 상태 다시 출력
if SWEEPS_EXTRA:
    print(f'[SWEEPS_EXTRA applied] final SWEEPS : {SWEEPS}')
    print(f'total runs now: {1 + sum(len(v) for v in SWEEPS.values())}')

# SWEEP_OUT_DIR 폴더명도 SWEEPS 확정 후 재계산 (태그 일치)
_sweep_tag = '-'.join(sorted(SWEEPS)) or 'baseline_only'
SWEEP_OUT_DIR = ENSEMBLE_ROOT / 'sweeps' / f'{_dt.datetime.now().strftime("%Y%m%d_%H%M%S")}_{_sweep_tag}'
SWEEP_OUT_DIR.mkdir(parents=True, exist_ok=True)
(SWEEP_OUT_DIR / 'config.json').write_text(json.dumps({
    'ensemble_root': str(ENSEMBLE_ROOT),
    'ARGS_parsed': vars(args),
    'SWEEPS_EXTRA': SWEEPS_EXTRA,
    'SWEEPS_final': SWEEPS, 'FIXED': FIXED, 'METRICS': METRICS,
    'TOP_N': TOP_N, 'RANK_METRIC': RANK_METRIC, 'RANK_ASCENDING': RANK_ASCENDING,
    'CLASSES': CLASSES, 'GPUS': GPUS, 'PARALLEL': PARALLEL,
}, indent=2, default=str))
print(f'\nFinal SWEEP_OUT_DIR → {SWEEP_OUT_DIR}')

# ======================================================================
#  wandb init (선택) — --wandb-project 가 있어야 실제 업로드
# ======================================================================
wandb_run = None
if args.wandb_project:
    import wandb
    wandb_run_name = args.wandb_run_name or f'{_dt.datetime.now().strftime("%Y%m%d_%H%M%S")}_{ENSEMBLE_ROOT.name}'
    wandb_run = wandb.init(
        project=args.wandb_project, name=wandb_run_name,
        mode=args.wandb_mode, reinit=True,
        config={**vars(args), 'ENSEMBLE_ROOT': str(ENSEMBLE_ROOT),
                'FIXED': FIXED, 'SWEEPS_final': SWEEPS},
    )
    print(f'\nwandb run: {wandb_run.name}  mode={args.wandb_mode}  project={args.wandb_project}')
else:
    print(f'\n[wandb disabled] --wandb-project 미설정 → 로컬 PNG 만 저장됨')


In [74]:
# %reload_ext autoreload

## 1. 한 config 실행 = 한 run

- VP-SGLD per-class (c0, c1) 를 내부에서 순차 실행 (한 run 안은 sequential 유지)
- `sample_quality`: `|real.mean - synth.mean| + |real.std - synth.std|` 평균 (낮을수록 좋음)
- `diagnostics`: 마지막 step 의 `M_mean`, `var_median`, `score_norm`

In [ ]:
# real 데이터 캐시 (매 run 에서 재로딩 방지)
REAL_BY_C = {c: np.load(ENSEMBLE_ROOT / f'c{c}' / 'class_data.npz')['X_class']
             for c in CLASSES}

SAMPLES_DIR = SWEEP_OUT_DIR / 'samples'
SAMPLES_DIR.mkdir(exist_ok=True)

# === trajectory 저장은 **항상 ON** (모든 step × 모든 chain) =================
# 분석/재시각화/M-vs-N scan 등 모두 trajectory 의존. 토글 X.
# 용량: T=50, N=500, d=9 → config 당 ~2MB compressed (negligible)
# =========================================================================

def resolve_config(overrides):
    '''FIXED 위에 overrides 를 덮어씌운 dict.'''
    return {**FIXED, **overrides}

def _tag_for(overrides):
    if not overrides:
        return 'baseline'
    (k, v), = overrides.items()
    return f'{k}_{v}'

def run_one_config(overrides, gpu):
    '''VP-SGLD 한 run + metric 계산 + 샘플/trajectory 저장.'''
    cfg = resolve_config(overrides)
    out = {**overrides}

    t0 = time.time()
    syn_by_c = {}
    traj_by_c = {}
    last_diag_by_c = {}
    all_diags_by_c = {}                          # 모든 step diag 저장
    for c in CLASSES:
        result = vp_sgld_from_ensemble(
            ENSEMBLE_ROOT / f'c{c}',
            n_samples=cfg['n_samples'], n_steps=cfg['n_steps'],
            beta=cfg['beta'], eta=cfg['eta'], tau=cfg['tau'],
            sigma_start=cfg['sigma_start'], auto_beta=cfg['auto_beta'],
            restart=cfg['restart'],
            kappa_sigma=cfg['kappa_sigma'], kappa_mu=cfg['kappa_mu'],
            ignore_variance=cfg.get('ignore_variance', False),
            seed=cfg['seed'], gpu=gpu,
            return_diagnostics=True,
            return_trajectory=True,
        )
        samples, diags, traj = result
        traj_by_c[c] = traj.numpy()              # (T+1, N, d)
        syn_by_c[c] = samples.numpy()
        last_diag_by_c[c] = diags[-1]
        all_diags_by_c[c] = diags                # 전체 step 의 diag 보관
    out['elapsed_sec'] = round(time.time() - t0, 2)

    # === 샘플 + (optional) trajectory 저장 ===
    tag = _tag_for(overrides)
    samples_path = SAMPLES_DIR / f'{tag}.npz'
    save_kv = {f'X_c{c}': syn_by_c[c] for c in CLASSES}
    X_concat = np.vstack([syn_by_c[c] for c in CLASSES])
    y_concat = np.concatenate([np.full(len(syn_by_c[c]), c) for c in CLASSES]).astype(np.int64)
    save_kv.update({'X': X_concat, 'y': y_concat,
                    'n_samples_per_class': np.array([len(syn_by_c[c]) for c in CLASSES]),
                    'cfg_json': np.array(json.dumps({**cfg, **overrides}), dtype=object)})
    for c, t in traj_by_c.items():
        save_kv[f'traj_c{c}'] = t.astype(np.float32)        # float32 로 용량 절반
    save_kv['n_steps'] = np.array(cfg['n_steps'])

    # === per-step diagnostics (모든 step 의 SGLD 텀 값들) ===
    _first_c = CLASSES[0]
    if all_diags_by_c[_first_c]:
        _diag_keys = [k for k in all_diags_by_c[_first_c][0].keys()
                       if isinstance(all_diags_by_c[_first_c][0][k], (int, float))]
        for c in CLASSES:
            diag_mat = np.array([[d.get(k, np.nan) for k in _diag_keys]
                                   for d in all_diags_by_c[c]], dtype=np.float32)
            save_kv[f'diag_c{c}'] = diag_mat     # (T, n_keys) float32
        save_kv['diag_cols'] = np.array(_diag_keys, dtype=object)

    np.savez_compressed(samples_path, **save_kv)
    out['samples_file'] = str(samples_path.relative_to(ENSEMBLE_ROOT))

    if 'sample_quality' in METRICS:
        diffs = []
        for c, s in syn_by_c.items():
            r = REAL_BY_C[c]
            diffs.append(float(np.abs(r.mean(0) - s.mean(0)).mean() +
                                np.abs(r.std(0)  - s.std(0)).mean()))
        out['sample_quality'] = float(np.mean(diffs))

    if 'diagnostics' in METRICS:
        out['M_mean']    = float(np.mean([last_diag_by_c[c]['M_mean']    for c in CLASSES]))
        out['var_median']= float(np.mean([last_diag_by_c[c]['var_median'] for c in CLASSES]))
        out['score_norm']= float(np.mean([last_diag_by_c[c]['score_norm'] for c in CLASSES]))

    return out

# baseline = FIXED 그대로
print('[baseline] running FIXED...')
baseline_row = run_one_config({}, gpu=GPUS[0])
baseline_row['axis'] = '_baseline'; baseline_row['value'] = None
print(f'  sample_quality = {baseline_row["sample_quality"]:.4f}  '
      f'→ samples at {baseline_row["samples_file"]}  ({baseline_row["elapsed_sec"]:.1f}s)')
baseline_row


## 2. OAT sweep — GPU 병렬

각 axis 의 값마다 한 run. 축 하나당 n 값 → `len(axis) × len(values)` runs.
`GPUS` 에 라운드로빈 분배 (각 run 은 독립 → threading 충분).

In [76]:
# 모든 sweep task 펼치기
all_tasks = []
for axis, values in SWEEPS.items():
    for v in values:
        all_tasks.append((axis, v, {axis: v}))

print(f'{len(all_tasks)} runs  |  PARALLEL={PARALLEL}  |  GPUS={GPUS}')

def _worker(task_gpu):
    axis, v, overrides, gpu = task_gpu
    out = run_one_config(overrides, gpu)
    out['axis'] = axis; out['value'] = v
    return out

assignments = [(*t, GPUS[i % len(GPUS)]) for i, t in enumerate(all_tasks)]

t0 = time.time()
results = []
if PARALLEL:
    with ThreadPoolExecutor(max_workers=len(GPUS)) as ex:
        for r in ex.map(_worker, assignments):
            results.append(r)
            print(f'  {r["axis"]}={r["value"]:<8}  sq={r["sample_quality"]:.4f}  M={r.get("M_mean",0):.3f}  ({r["elapsed_sec"]:.1f}s)')
else:
    # 단일 GPU 순차 실행
    for a in assignments:
        r = _worker(a)
        results.append(r)
        print(f'  {r["axis"]}={r["value"]:<8}  sq={r["sample_quality"]:.4f}  M={r.get("M_mean",0):.3f}  ({r["elapsed_sec"]:.1f}s)')

print(f'\nsweep total {time.time()-t0:.1f}s')


0 runs  |  PARALLEL=True  |  GPUS=[0, 1, 2, 3]

sweep total 0.0s


## 3. 결과 DataFrame + axis 별 plot

In [77]:
sweep_df = pd.DataFrame([baseline_row] + results)
display_cols = ['axis', 'value'] + [c for c in ['sample_quality','M_mean','var_median','score_norm'] if c in sweep_df.columns]
sweep_df[display_cols].sort_values(['axis', 'value'])

,axis,value,sample_quality,M_mean,var_median,score_norm
0,_baseline,None,1.5707,1.0,1.0000e-08,0.0002


In [78]:
# axis 당 line plot
axes_to_plot = [a for a in SWEEPS]
if axes_to_plot:
    n = len(axes_to_plot)
    fig, axs = plt.subplots(1, n, figsize=(5*n, 4))
    axs = np.atleast_1d(axs)
    for ax, axis_name in zip(axs, axes_to_plot):
        sub = sweep_df[sweep_df['axis']==axis_name].sort_values('value')
        ax.plot(sub['value'], sub['sample_quality'], 'o-', label='sample_quality')
        ax.axhline(baseline_row['sample_quality'], color='gray', ls=':', label='baseline (FIXED)')
        ax.set_xlabel(axis_name); ax.set_ylabel('sample_quality (↓)')
        ax.set_title(f'axis: {axis_name}')
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout()
    fig_path = SWEEP_OUT_DIR / 'sweep_lines.png'
    fig.savefig(fig_path, dpi=120, bbox_inches='tight')
    print(f'  saved: {fig_path}')
    plt.show()
else:
    print('SWEEPS 가 비어있음 — plot 생략.')


SWEEPS 가 비어있음 — plot 생략.


## 4. Top-N config 선정 + nb 03 용 JSON 저장

In [79]:
# sweep 결과만 정렬 (baseline 제외)
sweep_only = sweep_df[sweep_df['axis'] != '_baseline'].copy()
if len(sweep_only):
    top = sweep_only.sort_values(RANK_METRIC, ascending=RANK_ASCENDING).head(TOP_N).reset_index(drop=True)
else:
    # SWEEPS 비어있으면 baseline 만 top 에 넣음
    top = sweep_df[sweep_df['axis'] == '_baseline'].copy().reset_index(drop=True)
print(f'Top {len(top)} by {RANK_METRIC} ({"asc" if RANK_ASCENDING else "desc"}):')
print(top[display_cols])

# 각 top row 의 full config 만들기 (FIXED + override)
top_configs = []
for _, row in top.iterrows():
    axis = row['axis']
    override = {} if axis == '_baseline' else {axis: row['value']}
    full_cfg = resolve_config(override)
    full_cfg['_axis'] = axis; full_cfg['_value'] = row['value']
    full_cfg['_sample_quality'] = float(row['sample_quality'])
    top_configs.append(full_cfg)

TOP_CONFIGS = top_configs   # 대문자 alias — 이후 viz 셀이 참조

save_data = {
    'ensemble_root': str(ENSEMBLE_ROOT),
    'sweep_out_dir': str(SWEEP_OUT_DIR),
    'top_n': TOP_N,
    'rank_metric': RANK_METRIC,
    'rank_ascending': RANK_ASCENDING,
    'baseline_sample_quality': float(baseline_row['sample_quality']),
    'configs': top_configs,
}

# Primary: sweep 전용 폴더
primary_path = SWEEP_OUT_DIR / 'sweep_top.json'
primary_path.write_text(json.dumps(save_data, indent=2, default=str))
print(f'\nSaved (primary)  → {primary_path}')

# Mirror: ENSEMBLE_ROOT 루트에도 복사 — nb 03 이 최신을 자동 로드하도록
mirror_path = ENSEMBLE_ROOT / 'sweep_top.json'
mirror_path.write_text(json.dumps(save_data, indent=2, default=str))
print(f'Mirror (for nb 03) → {mirror_path}')


Top 1 by sample_quality (asc):
        axis value  sample_quality  M_mean  var_median  score_norm
0  _baseline  None          1.5707     1.0  1.0000e-08      0.0002

Saved (primary)  → experiments/ebms/20260415_214026_Distance_EBM/sweeps/20260416_021357_baseline_only/sweep_top.json
Mirror (for nb 03) → experiments/ebms/20260415_214026_Distance_EBM/sweep_top.json


## 5. Full sweep 결과 + 디렉토리 요약 저장

In [80]:
csv_path = SWEEP_OUT_DIR / 'sweep_full.csv'
sweep_df.to_csv(csv_path, index=False)
print(f'full sweep saved → {csv_path}  ({len(sweep_df)} rows)')

# 디렉토리 전체 요약
print(f'\n=== {SWEEP_OUT_DIR} ===')
for p in sorted(SWEEP_OUT_DIR.iterdir()):
    print(f'  {p.name:<24}  {p.stat().st_size:>8} bytes')


full sweep saved → experiments/ebms/20260415_214026_Distance_EBM/sweeps/20260416_021357_baseline_only/sweep_full.csv  (1 rows)

=== experiments/ebms/20260415_214026_Distance_EBM/sweeps/20260416_021357_baseline_only ===
  config.json                   1370 bytes
  samples                       4096 bytes
  sweep_full.csv                 220 bytes
  sweep_top.json                 700 bytes


## 7. Inline 최종 시각화 (matplotlib, sweep 완료 후)

HTML 없이 **notebook 안에서 바로** 보는 정적 플롯. 각 Top-N config × class 에 대해:
- 배경: ensemble std contourf (nb 01 캐시)
- real positives (cyan) + surrogate negatives (검은 ×)
- **final generated samples** — 모든 N 체인의 마지막 위치 (회색 dot)
- **trajectory 선 + 이동방향 arrow** (x_0 → x_T) for `MPL_CHAINS_SHOW` 개 체인
- 애니메이션 / 슬라이더 없음 — 한 장짜리 정적 figure 로 기록용.

In [86]:
from experiments.viz_trajectory import (
    plot_trajectory_summary_mpl,          # SGLD 재실행해서 trajectory 계산
    plot_trajectory_from_saved_mpl,       # 저장된 npz 에서 trajectory 로드
)


def _resolve_viz_sweep_bundle(ensemble_root, sweep_dir_override=None, prefer_current=False):
    ensemble_root = Path(ensemble_root)
    sweeps_root = ensemble_root / 'sweeps'
    candidates = []
    if sweeps_root.exists():
        candidates = sorted(
            p for p in sweeps_root.iterdir()
            if p.is_dir() and (p / 'sweep_top.json').exists()
        )

    current_sweep = globals().get('SWEEP_OUT_DIR')
    if sweep_dir_override is not None:
        sweep_dir = Path(sweep_dir_override)
    elif prefer_current and current_sweep is not None and (Path(current_sweep) / 'sweep_top.json').exists():
        sweep_dir = Path(current_sweep)
    elif candidates:
        sweep_dir = candidates[-1]
    else:
        raise FileNotFoundError(
            f'No sweep directory with sweep_top.json under {sweeps_root}. '
            'Run nb 02 sweep first or set VIZ_SWEEP_DIR_OVERRIDE.'
        )

    sweep_top_path = sweep_dir / 'sweep_top.json'
    assert sweep_top_path.exists(), f'No sweep_top.json at {sweep_top_path}'
    sweep_top = json.loads(sweep_top_path.read_text())
    viz_ensemble_root = Path(sweep_top.get('ensemble_root', ensemble_root))
    configs = sweep_top['configs']
    return sweep_dir, sweep_top_path, viz_ensemble_root, configs


# --- 어떤 sweep 결과를 시각화할지 선택 -------------------------------------
VIZ_SWEEP_DIR_OVERRIDE = None     # 예: ENSEMBLE_ROOT / 'sweeps' / '20260416_015927_baseline_only'
VIZ_PREFER_CURRENT_SWEEP_OUT_DIR = False
# False 이면 override 없을 때 ENSEMBLE_ROOT/sweeps/ 중 가장 최근 sweep 사용
# True  이면 현재 세션의 SWEEP_OUT_DIR 를 우선 사용

VIZ_SWEEP_DIR, VIZ_SWEEP_TOP_PATH, VIZ_ENSEMBLE_ROOT, VIZ_TOP_CONFIGS = _resolve_viz_sweep_bundle(
    ENSEMBLE_ROOT,
    sweep_dir_override=VIZ_SWEEP_DIR_OVERRIDE,
    prefer_current=VIZ_PREFER_CURRENT_SWEEP_OUT_DIR,
)

# --- inline mpl 파라미터 ------------------------------------------------
USE_SAVED_TRAJECTORY = False       # True: samples/*.npz 의 traj_c{c} 로드 (빠름, SGLD 재실행 X)
                                    # False: SGLD 재실행 후 plot (오래 걸림)
MPL_N              = 3            # Top-N 중 몇 개 config 에 대해 그릴지 (None=전체)
MPL_CLASSES        = CLASSES
MPL_CHAINS_SHOW    = 20           # trajectory + arrow 그릴 체인 수
MPL_TRAJ_CMAP      = 'turbo'
MPL_BG_CMAP        = 'magma'
MPL_BACKGROUND     = 'std'        # 'std' | 'mean'
MPL_NEG_COLOR      = 'black'
MPL_REAL_COLOR     = 'cyan'
MPL_FINAL_COLOR    = 'red'
MPL_FIGSIZE        = (9, 7)
MPL_SHOW_LINES     = False        # trajectory 선. False 면 arrow 만
MPL_SHOW_ARROWS    = False         # x_0 → x_T 방향 화살표
# -------------------------------------------------------------------------

VIZ_SWEEP_DIR = 'experiments/ebms/20260415_214026_Distance_EBM/sweeps/20260416_004213_baseline_only'

configs_to_plot = VIZ_TOP_CONFIGS[:MPL_N] if MPL_N else VIZ_TOP_CONFIGS
print(f'[viz] sweep dir     : {VIZ_SWEEP_DIR}')
print(f'[viz] sweep_top.json: {VIZ_SWEEP_TOP_PATH}')
print(f'[viz] ensemble root : {VIZ_ENSEMBLE_ROOT}')
print(f'{len(configs_to_plot)} config × {len(MPL_CLASSES)} class = '
      f'{len(configs_to_plot) * len(MPL_CLASSES)} figures inline')
print(f'mode: {"저장된 trajectory 로드" if USE_SAVED_TRAJECTORY else "SGLD 재실행"}')

final_samples = {}  # {(ci, c): (N, 2) final PCA coords}
for ci, cfg in enumerate(configs_to_plot):
    # 이 config 에 해당하는 samples/*.npz 경로 찾기
    tag = 'baseline' if cfg['_axis'] == '_baseline' else f"{cfg['_axis']}_{cfg['_value']}"
    samples_npz = VIZ_SWEEP_DIR / 'samples' / f'{tag}.npz'
    label = 'baseline' if cfg['_axis'] == '_baseline' else f"{cfg['_axis']}={cfg['_value']}"

    for c in MPL_CLASSES:
        var_mode = 'off' if cfg.get('ignore_variance', False) else 'on'
        title = (f"[{label}]  ·  c{c}  ·  sq_nb02={cfg['_sample_quality']:.3f}  ·  var={var_mode}\n"
                 f"β={cfg['beta']}, η={cfg['eta']}, τ={cfg['tau']}, "
                 f"σ_start={cfg['sigma_start']}, T={cfg['n_steps']}, N={cfg['n_samples']}")

        if USE_SAVED_TRAJECTORY and samples_npz.exists():
            fig, ax, final = plot_trajectory_from_saved_mpl(
                samples_npz_path=samples_npz,
                ensemble_root=VIZ_ENSEMBLE_ROOT, class_c=c,
                n_chains_show=MPL_CHAINS_SHOW,
                background=MPL_BACKGROUND, bg_cmap=MPL_BG_CMAP,
                traj_cmap=MPL_TRAJ_CMAP, real_color=MPL_REAL_COLOR,
                neg_color=MPL_NEG_COLOR, final_color=MPL_FINAL_COLOR,
                figsize=MPL_FIGSIZE, title=title, verbose=False,
                show_trajectory_lines=MPL_SHOW_LINES,
                show_trajectory_arrows=MPL_SHOW_ARROWS,
                gpu=GPUS[0], seed=cfg['seed'],
            )
        else:
            if USE_SAVED_TRAJECTORY:
                print(f'  [warn] {samples_npz} 없음 — SGLD 재실행으로 폴백')
            fig, ax, final = plot_trajectory_summary_mpl(
                ensemble_root=VIZ_ENSEMBLE_ROOT, class_c=c,
                n_samples=cfg['n_samples'], n_steps=cfg['n_steps'],
                beta=cfg['beta'], eta=cfg['eta'], tau=cfg['tau'],
                sigma_start=cfg['sigma_start'], auto_beta=cfg['auto_beta'],
                restart=cfg['restart'],
                kappa_sigma=cfg.get('kappa_sigma'), kappa_mu=cfg.get('kappa_mu'),
                ignore_variance=cfg.get('ignore_variance', False),
                seed=cfg['seed'], gpu=GPUS[0],
                n_chains_show=MPL_CHAINS_SHOW,
                background=MPL_BACKGROUND, bg_cmap=MPL_BG_CMAP,
                traj_cmap=MPL_TRAJ_CMAP, real_color=MPL_REAL_COLOR,
                neg_color=MPL_NEG_COLOR, final_color=MPL_FINAL_COLOR,
                figsize=MPL_FIGSIZE, title=title, verbose=False,
                show_trajectory_lines=MPL_SHOW_LINES,
                show_trajectory_arrows=MPL_SHOW_ARROWS,
            )

        png_path = VIZ_SWEEP_DIR / 'viz' / f'inline_c{c}_{label}.png'
        png_path.parent.mkdir(exist_ok=True, parents=True)
        fig.savefig(png_path, dpi=130, bbox_inches='tight')
        if wandb_run is not None:
            wandb_run.log({f'inline_final/c{c}/{label}': wandb.Image(fig)})
        plt.show()
        final_samples[(ci, c)] = final
        print(f'  saved: {png_path}' + ('  (+ wandb)' if wandb_run is not None else ''))

print(f'\n{len(final_samples)} inline figures done.')


[viz] sweep dir     : experiments/ebms/20260415_214026_Distance_EBM/sweeps/20260416_004213_baseline_only
[viz] sweep_top.json: experiments/ebms/20260415_214026_Distance_EBM/sweeps/20260416_021357_baseline_only/sweep_top.json
[viz] ensemble root : experiments/ebms/20260415_214026_Distance_EBM
1 config × 2 class = 2 figures inline
mode: SGLD 재실행


TypeError: unsupported operand type(s) for /: 'str' and 'str'

## 8. Figure C — per-step chain density evolution (논문 style)

각 Top-N config × class 에 대해 **4-panel 1x4 grid** 로 체인이 시간에 따라
어떻게 수렴하는지 density 로 보여줌. `samples/*.npz` 의 `traj_c{c}` 재사용.

- **배경**: ensemble std (차분한 grayscale, 방해 안 되게)
- **overlay**: 해당 step 의 체인 positions 2D-KDE contourf (푸른 density)
- panel 간 **density scale 동일** → 시간에 따라 퍼지고/수렴하는 정도 직접 비교
- real (cyan) / negatives (red +) 는 모든 panel 에 anchor 로

In [ ]:
from experiments.viz_trajectory import plot_trajectory_evolution_mpl

if '_resolve_viz_sweep_bundle' not in globals():
    def _resolve_viz_sweep_bundle(ensemble_root, sweep_dir_override=None, prefer_current=False):
        ensemble_root = Path(ensemble_root)
        sweeps_root = ensemble_root / 'sweeps'
        candidates = []
        if sweeps_root.exists():
            candidates = sorted(
                p for p in sweeps_root.iterdir()
                if p.is_dir() and (p / 'sweep_top.json').exists()
            )

        current_sweep = globals().get('SWEEP_OUT_DIR')
        if sweep_dir_override is not None:
            sweep_dir = Path(sweep_dir_override)
        elif prefer_current and current_sweep is not None and (Path(current_sweep) / 'sweep_top.json').exists():
            sweep_dir = Path(current_sweep)
        elif candidates:
            sweep_dir = candidates[-1]
        else:
            raise FileNotFoundError(
                f'No sweep directory with sweep_top.json under {sweeps_root}. '
                'Run nb 02 sweep first or set VIZ_SWEEP_DIR_OVERRIDE.'
            )

        sweep_top_path = sweep_dir / 'sweep_top.json'
        assert sweep_top_path.exists(), f'No sweep_top.json at {sweep_top_path}'
        sweep_top = json.loads(sweep_top_path.read_text())
        viz_ensemble_root = Path(sweep_top.get('ensemble_root', ensemble_root))
        configs = sweep_top['configs']
        return sweep_dir, sweep_top_path, viz_ensemble_root, configs


VIZ_SWEEP_DIR_OVERRIDE = globals().get('VIZ_SWEEP_DIR_OVERRIDE', None)
VIZ_PREFER_CURRENT_SWEEP_OUT_DIR = globals().get('VIZ_PREFER_CURRENT_SWEEP_OUT_DIR', False)
VIZ_SWEEP_DIR, VIZ_SWEEP_TOP_PATH, VIZ_ENSEMBLE_ROOT, VIZ_TOP_CONFIGS = _resolve_viz_sweep_bundle(
    ENSEMBLE_ROOT,
    sweep_dir_override=VIZ_SWEEP_DIR_OVERRIDE,
    prefer_current=VIZ_PREFER_CURRENT_SWEEP_OUT_DIR,
)

# --- figure C 파라미터 -------------------------------------------------
EVO_N               = 3          # Top-N 중 몇 개 config
EVO_CLASSES         = CLASSES
EVO_STEPS_TO_SHOW   = None       # None = [0, T/3, 2T/3, T] 자동. list 로 커스텀: [0, 10, 30, 50]
EVO_BACKGROUND      = 'std'      # 'std' | 'mean'
EVO_BG_CMAP         = 'Greys'    # 배경 (차분)
EVO_DENSITY_CMAP    = 'Blues'    # density overlay (강조)
EVO_DENSITY_ALPHA   = 0.75
EVO_DENSITY_METHOD  = 'kde'      # 'kde' | 'hist2d'
EVO_DENSITY_GRID    = 100
EVO_DENSITY_LEVELS  = 8
EVO_FIGSIZE         = None       # None = 자동 (panel 당 ~3.3인치)
EVO_SHOW_REAL       = True
EVO_SHOW_NEG        = True
EVO_SHOW_DENSITY    = True        # chain 2D-density contour 표시
EVO_SHOW_SCATTER    = False       # 추가로 체인 점 scatter 도 (density 와 중복, 디버그용)
# -------------------------------------------------------------------------

configs_to_plot = VIZ_TOP_CONFIGS[:EVO_N] if EVO_N else VIZ_TOP_CONFIGS
print(f'[evo] sweep dir     : {VIZ_SWEEP_DIR}')
print(f'[evo] sweep_top.json: {VIZ_SWEEP_TOP_PATH}')
print(f'[evo] ensemble root : {VIZ_ENSEMBLE_ROOT}')
print(f'Figure C: {len(configs_to_plot)} config × {len(EVO_CLASSES)} class')

evo_figs = []
for ci, cfg in enumerate(configs_to_plot):
    tag = 'baseline' if cfg['_axis'] == '_baseline' else f"{cfg['_axis']}_{cfg['_value']}"
    label = 'baseline' if cfg['_axis'] == '_baseline' else f"{cfg['_axis']}={cfg['_value']}"
    samples_npz = VIZ_SWEEP_DIR / 'samples' / f'{tag}.npz'

    if not samples_npz.exists():
        print(f'  [skip] {tag} npz 없음 — {VIZ_SWEEP_DIR / "samples"} 확인')
        continue
    data = np.load(samples_npz, allow_pickle=True)

    for c in EVO_CLASSES:
        traj_key = f'traj_c{c}'
        if traj_key not in data.files:
            print(f'  [skip] {tag} c{c}: traj_c{c} 없음 (레거시 — trajectory 미저장 npz)')
            continue
        traj = data[traj_key]   # (T+1, N, d)
        ig = cfg.get("ignore_variance", False)
        var_mode = "off" if ig else "on"
        title = (f"[{label}]  class {c}  ·  var={var_mode}\n"
                 f"β={cfg['beta']}, η={cfg['eta']}, τ={cfg['tau']}, "
                 f"σ_start={cfg['sigma_start']}, T={cfg['n_steps']}, N={cfg['n_samples']}, "
                 f"auto_β={cfg['auto_beta']}, ignore_var={ig}")
        fig, axes = plot_trajectory_evolution_mpl(
            traj=traj, ensemble_root=VIZ_ENSEMBLE_ROOT, class_c=c,
            steps_to_show=EVO_STEPS_TO_SHOW,
            background=EVO_BACKGROUND,
            bg_cmap=EVO_BG_CMAP,
            density_cmap=EVO_DENSITY_CMAP,
            density_alpha=EVO_DENSITY_ALPHA,
            density_method=EVO_DENSITY_METHOD,
            density_grid=EVO_DENSITY_GRID,
            density_levels=EVO_DENSITY_LEVELS,
            figsize=EVO_FIGSIZE,
            title=title, gpu=GPUS[0],
            show_real=EVO_SHOW_REAL, show_neg=EVO_SHOW_NEG,
            show_chain_density=EVO_SHOW_DENSITY,
            show_chain_scatter=EVO_SHOW_SCATTER,
            verbose=False,
        )
        png_path = VIZ_SWEEP_DIR / 'viz' / f'evolution_c{c}_{label}.png'
        png_path.parent.mkdir(exist_ok=True, parents=True)
        fig.savefig(png_path, dpi=150, bbox_inches='tight')
        if wandb_run is not None:
            wandb_run.log({f'evolution/c{c}/{label}': wandb.Image(fig)})
        plt.show()
        evo_figs.append(png_path)
        print(f'  saved: {png_path}' + ('  (+ wandb)' if wandb_run is not None else ''))

print(f'\n{len(evo_figs)} evolution figures done.')

# wandb 활성화면 session 종료
if wandb_run is not None:
    wandb_run.finish()
